# AXUM Restoration Model Ablation
### Ge'ez Restoration: 1.5B/3B/7B (Qwen2.5) vs 2B/4B (Qwen3.5), with RAG (BM25) vs fine-tune-only, and thinking-mode on/off for Qwen3.5

**How to use this notebook:**
1. Runtime -> Change runtime type -> select **T4 GPU** (free tier).
2. Run cells top to bottom, in order, once: Setup, Upload Corpus, Augmentation, Retrieval, Fixed Eval Set.
3. Then run **one model section at a time** (Section 5.1 through 5.5). Each is self-contained: loads its base model, fine-tunes, scores, saves results, frees memory. If Colab disconnects, just re-run from whichever model section you were on -- no need to restart from the top (the corpus/eval-set cells save to disk).
4. Run the final **Results Summary** cell after all five models have been scored.

No files needed except your Ge'ez corpus text file, uploaded in Section 2. Everything else downloads automatically from Hugging Face inside this notebook -- nothing to download locally first.


## 1. Setup -- run once

In [ ]:
# Core libraries. bitsandbytes = 4-bit quantization (QLoRA), peft = LoRA adapters,
# trl = simplified fine-tuning loop, rank_bm25 = the BM25 retrieval we already use in production,
# python-Levenshtein = fast edit-distance scoring.
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets rank_bm25 python-Levenshtein sentencepiece

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only -- check Runtime > Change runtime type > T4 GPU")


# LOCAL_WORK_DIR replaces every hardcoded /content/... path below.
# Works the same in Colab or locally -- Colab's /content/ was never
# anything but a writable working directory, this is just an explicit
# one instead of an implicit Colab-only convention.
from pathlib import Path
LOCAL_WORK_DIR = Path("ablation_workdir")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_WORK_DIR / "checkpoints").mkdir(exist_ok=True)


## 2. Upload your Ge'ez corpus

Upload a plain `.txt` file: **one passage/inscription per line**, UTF-8 encoded, Ge'ez script text only
(Kebra Nagast excerpts, Book of Enoch Ge'ez, known royal inscriptions -- whatever you've been using so far).

If you don't have this ready and just want to test the pipeline mechanics first, the fallback cell below
generates a small dummy placeholder corpus so you can verify everything runs before plugging in real data.


In [ ]:
# Colab-only upload flow replaced with a direct local path (Systems
# Integration Engineer, this pass) -- `from google.colab import files`
# only exists inside Colab and would crash immediately on any local
# machine. Point CORPUS_PATH at your real corpus file; if it doesn't
# exist, the next cell falls back to its existing placeholder (same
# behavior as before, just triggered by a missing file instead of a
# skipped upload dialog).
CORPUS_PATH = "data/restoration_corpus_chunked.json"  # <-- point this at your real corpus file

if CORPUS_PATH.exists():
    corpus_filename = str(CORPUS_PATH)
    print(f"Found corpus file: {corpus_filename}")
else:
    corpus_filename = None
    print(f"No corpus file at {CORPUS_PATH} -- will use placeholder corpus in the next cell.")
    print("Set CORPUS_PATH above to your real corpus file and re-run this cell to use it.")


In [ ]:
# Load the corpus. Falls back to a small placeholder if nothing was uploaded above,
# purely so you can test the notebook mechanics end to end before your real corpus is ready.

if corpus_filename and os.path.exists(corpus_filename):
    with open(corpus_filename, "r", encoding="utf-8") as f:
        corpus_lines = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(corpus_lines)} lines from {corpus_filename}")
else:
    # PLACEHOLDER ONLY -- replace by uploading a real file above before trusting any results.
    corpus_lines = [
        "\u1265\u1235\u121d \u12a0\u1265 \u12c8\u12c8\u120d\u12f5 \u12c8\u12c8\u1218\u1295\u134d\u1235 \u1245\u12f1\u1235",
        "\u12ea\u1218\u1295 \u122d\u1275\u12d5 \u12a5\u1235\u12ad \u12a5\u130b\u12dc\u12a4\u122d \u1235\u121b\u12ed",
        "\u12c8\u12eb\u12a5\u121d\u122d\u1275 \u1218\u12ab\u1295 \u12a0\u12eb\u1275 \u12d3\u1208\u121d",
    ] * 20  # repeated only so downstream code has enough lines to slice a train/eval split for a mechanics test
    print(f"WARNING: using {len(corpus_lines)} PLACEHOLDER lines. Upload a real corpus before trusting any results.")


## 3. Damage-simulation augmentation

Ported from the Aeneas/DeepMind restoration-model approach (see the augmentation guide sent to AI/CV Engineer).
Two damage types per sample, generated fresh every time this function is called -- never cache these to a static file,
that's what turns a small corpus into effectively unlimited distinct training examples.


In [ ]:
import numpy as np

CHAR_MASK = "\u2043"        # known-length gap: one placeholder per missing character
UNKNOWN_GAP = "[MISSING]"    # unknown-length gap: ONE placeholder regardless of underlying length

def random_mask_span(text, geometric_p=0.1, limit_chars=None):
    if geometric_p and limit_chars:
        span_len = min(np.random.geometric(geometric_p), limit_chars)
    elif geometric_p:
        span_len = np.random.geometric(geometric_p)
    else:
        span_len = limit_chars
    if span_len <= 0 or span_len > len(text):
        return []
    start = np.random.randint(0, len(text) - span_len + 1)
    return list(range(start, start + span_len))

def inject_unknown_length_gap(text, geometric_p=0.25, min_len=None):
    span_len = np.random.geometric(geometric_p) - 1
    if span_len == 0 or (min_len and len(text) - span_len < min_len):
        return text, 0
    start = np.random.randint(0, len(text) - span_len + 1)
    end = start + span_len
    return text[:start] + UNKNOWN_GAP + text[end:], span_len

def generate_training_pair(
    clean_text,
    char_mask_rate_min=0.0,
    char_mask_rate_max=0.5,     # lower than Aeneas's 0.75 default -- our passages are likely shorter, see guide
    span_mask_ratio=0.15,
    span_mask_geometric_p=0.1,
    inject_unknown_p=0.25,
    min_context_len=10,
):
    """Given one clean line, produce one (damaged_input, target) pair. Call fresh every epoch -- do not cache."""
    target = clean_text
    if len(clean_text) < min_context_len:
        return clean_text, target  # too short to safely damage, return as-is

    damaged, _ = inject_unknown_length_gap(clean_text, geometric_p=inject_unknown_p, min_len=min_context_len)

    char_mask_rate = np.random.uniform(char_mask_rate_min, char_mask_rate_max)
    maskable_idx = [i for i, c in enumerate(damaged) if damaged[i:i+len(UNKNOWN_GAP)] != UNKNOWN_GAP]
    mask_total = int(char_mask_rate * len(maskable_idx))
    mask_span_n = int(mask_total * span_mask_ratio)
    mask_char_n = mask_total - mask_span_n

    mask_idx = set()
    if mask_char_n > 0 and maskable_idx:
        mask_idx.update(np.random.choice(maskable_idx, min(mask_char_n, len(maskable_idx)), replace=False))
    if mask_span_n > 0:
        span_idx = random_mask_span(damaged, geometric_p=span_mask_geometric_p, limit_chars=mask_span_n)
        mask_idx.update(span_idx)

    damaged_chars = list(damaged)
    for i in mask_idx:
        if i < len(damaged_chars):
            damaged_chars[i] = CHAR_MASK
    damaged = "".join(damaged_chars)
    return damaged, target

# Quick sanity check
sample_damaged, sample_target = generate_training_pair(corpus_lines[0])
print("Example damaged: ", sample_damaged)
print("Example target:  ", sample_target)


## 4. Retrieval setup (BM25 -- matches the production pipeline's current decision)

This mirrors AI/CV Engineer's BM25-only retrieval choice, so the ablation numbers are actually representative
of what will run in production. Given one damaged passage, retrieve the most similar known passages from the
corpus to use as grounding context for restoration.


In [ ]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    # Character-level tokenization -- reasonable default for a script without whitespace-delimited words
    # in the way Latin scripts have; revisit if word-level segmentation becomes available for Ge'ez.
    return list(text)

bm25_corpus_tokens = [simple_tokenize(line) for line in corpus_lines]
bm25 = BM25Okapi(bm25_corpus_tokens)

def retrieve_similar(damaged_text, k=3):
    query_tokens = simple_tokenize(damaged_text.replace(UNKNOWN_GAP, "").replace(CHAR_MASK, ""))
    scores = bm25.get_scores(query_tokens)
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [corpus_lines[i] for i in top_k_idx]

# Sanity check
print("Retrieved parallels for the damaged example above:")
for p in retrieve_similar(sample_damaged):
    print(" -", p)


## 5. Fixed evaluation set -- generate ONCE, reuse for every model

This is important: all five models must be scored against the *same* damaged examples, or the comparison
is meaningless. Seeded random generation here, saved to disk, loaded (not regenerated) by every model section below.


In [ ]:
import json as _json

EVAL_SET_PATH = str(LOCAL_WORK_DIR / "fixed_eval_set.json")  # was /content/, Colab-only
N_EVAL_EXAMPLES = min(40, len(corpus_lines))

if not os.path.exists(EVAL_SET_PATH):
    np.random.seed(42)  # fixed seed -- this exact eval set must be identical across all five model runs
    eval_source_lines = list(np.random.choice(corpus_lines, N_EVAL_EXAMPLES, replace=False))
    eval_set = []
    for line in eval_source_lines:
        damaged, target = generate_training_pair(line)
        retrieved = retrieve_similar(damaged, k=3)
        eval_set.append({"damaged": damaged, "target": target, "retrieved": retrieved})
    with open(EVAL_SET_PATH, "w", encoding="utf-8") as f:
        _json.dump(eval_set, f, ensure_ascii=False, indent=2)
    print(f"Generated and saved {len(eval_set)} fixed eval examples to {EVAL_SET_PATH}")
else:
    with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
        eval_set = _json.load(f)
    print(f"Loaded existing fixed eval set ({len(eval_set)} examples) from {EVAL_SET_PATH} -- NOT regenerated, as intended.")


## 6. Shared training + scoring functions

Used identically by all five model sections below -- the model name/config is the only thing that changes.


In [ ]:
import Levenshtein
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import gc

RESULTS_PATH = str(LOCAL_WORK_DIR / "ablation_results.json")  # was /content/, Colab-only

def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r", encoding="utf-8") as f:
            return _json.load(f)
    return {}

def save_result(key, result_dict):
    results = load_results()
    results[key] = result_dict
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        _json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved result for '{key}'")

def build_prompt(damaged_text, retrieved_parallels, use_rag=True):
    """Chat-template format, NOT raw FIM tags -- see reasoning sent to AI/CV Engineer:
    the restoration model may double as the arm's instruction-parser, so we keep it
    compatible with normal chat-style fine-tuning rather than teaching a brand-new
    structural convention the base model has no pretrained representation for."""
    if use_rag and retrieved_parallels:
        context = "\n".join(f"- {p}" for p in retrieved_parallels)
        user_msg = (
            f"Similar known Ge\'ez passages for reference:\n{context}\n\n"
            f"Restore the following damaged Ge\'ez text. "
            f"'{CHAR_MASK}' marks single missing characters, '{UNKNOWN_GAP}' marks a gap of unknown length:\n"
            f"{damaged_text}"
        )
    else:
        user_msg = (
            f"Restore the following damaged Ge\'ez text. "
            f"'{CHAR_MASK}' marks single missing characters, '{UNKNOWN_GAP}' marks a gap of unknown length:\n"
            f"{damaged_text}"
        )
    return [{"role": "user", "content": user_msg}]

def make_training_dataset(tokenizer, n_examples=200, use_rag=True):
    """Regenerates masks fresh -- this is the "never cache" principle from the augmentation guide."""
    rows = []
    for _ in range(n_examples):
        line = np.random.choice(corpus_lines)
        damaged, target = generate_training_pair(line)
        retrieved = retrieve_similar(damaged, k=3) if use_rag else []
        messages = build_prompt(damaged, retrieved, use_rag=use_rag) + [{"role": "assistant", "content": target}]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        rows.append({"text": text})
    return Dataset.from_list(rows)

def score_model(model, tokenizer, use_rag=True, enable_thinking=None, max_new_tokens=128):
    """Edit-distance scoring against the FIXED eval set (Section 5) -- identical for every model."""
    total_dist = 0
    total_len = 0
    examples_out = []
    for ex in eval_set:
        retrieved = ex["retrieved"] if use_rag else []
        messages = build_prompt(ex["damaged"], retrieved, use_rag=use_rag)
        template_kwargs = {}
        if enable_thinking is not None:
            template_kwargs["enable_thinking"] = enable_thinking
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, **template_kwargs
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        dist = Levenshtein.distance(generated, ex["target"])
        total_dist += dist
        total_len += max(len(ex["target"]), 1)
        examples_out.append({"damaged": ex["damaged"], "target": ex["target"], "predicted": generated, "edit_distance": dist})
    normalized_edit_distance = total_dist / total_len  # lower is better, 0 = perfect
    return normalized_edit_distance, examples_out

def free_gpu_memory(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory freed.")


---
## 7. Run each candidate

Run these five sections **one at a time**. Each is self-contained (loads model, fine-tunes with QLoRA,
scores against the fixed eval set, saves the result, frees memory) so a disconnect only costs you the
model that was in progress, not everything before it.


### 5.1 Qwen2.5-1.5B-Instruct + RAG

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_KEY = "qwen2.5-1.5b-rag"
USE_RAG = True

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)

train_dataset = make_training_dataset(tokenizer, n_examples=200, use_rag=USE_RAG)

# output_dir was hardcoded to /content/checkpoints/... (Colab-only) -- now LOCAL_WORK_DIR
sft_config = SFTConfig(output_dir=str(LOCAL_WORK_DIR / "checkpoints" / RUN_KEY), num_train_epochs=3, per_device_train_batch_size=2,
                        gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=10, save_strategy="no",
                        report_to="none", max_seq_length=1024, dataset_text_field="text")
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, tokenizer=tokenizer)
trainer.train()

score, examples_out = score_model(model, tokenizer, use_rag=USE_RAG)
print(f"{RUN_KEY} normalized edit distance: {score:.4f} (lower is better)")
save_result(RUN_KEY, {"model": MODEL_NAME, "use_rag": USE_RAG, "normalized_edit_distance": score,
                       "n_train_examples": 200, "examples": examples_out[:5]})

free_gpu_memory(model, trainer)


### 5.2 Qwen2.5-3B-Instruct + RAG

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
RUN_KEY = "qwen2.5-3b-rag"
USE_RAG = True

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)

train_dataset = make_training_dataset(tokenizer, n_examples=200, use_rag=USE_RAG)

# output_dir was hardcoded to /content/checkpoints/... (Colab-only) -- now LOCAL_WORK_DIR
sft_config = SFTConfig(output_dir=str(LOCAL_WORK_DIR / "checkpoints" / RUN_KEY), num_train_epochs=3, per_device_train_batch_size=2,
                        gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=10, save_strategy="no",
                        report_to="none", max_seq_length=1024, dataset_text_field="text")
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, tokenizer=tokenizer)
trainer.train()

score, examples_out = score_model(model, tokenizer, use_rag=USE_RAG)
print(f"{RUN_KEY} normalized edit distance: {score:.4f} (lower is better)")
save_result(RUN_KEY, {"model": MODEL_NAME, "use_rag": USE_RAG, "normalized_edit_distance": score,
                       "n_train_examples": 200, "examples": examples_out[:5]})

free_gpu_memory(model, trainer)


### 5.3 Qwen2.5-7B-Instruct, fine-tune only (no RAG) -- the original baseline this ablation exists to test against

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
RUN_KEY = "qwen2.5-7b-noRAG"
USE_RAG = False

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)

# Smaller batch size than the 1.5B/3B runs -- 7B is a bigger memory footprint even at 4-bit on a T4.
train_dataset = make_training_dataset(tokenizer, n_examples=200, use_rag=USE_RAG)

# output_dir was hardcoded to /content/checkpoints/... (Colab-only) -- now LOCAL_WORK_DIR
sft_config = SFTConfig(output_dir=str(LOCAL_WORK_DIR / "checkpoints" / RUN_KEY), num_train_epochs=3, per_device_train_batch_size=1,
                        gradient_accumulation_steps=8, learning_rate=2e-4, logging_steps=10, save_strategy="no",
                        report_to="none", max_seq_length=1024, dataset_text_field="text")
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, tokenizer=tokenizer)
trainer.train()

score, examples_out = score_model(model, tokenizer, use_rag=USE_RAG)
print(f"{RUN_KEY} normalized edit distance: {score:.4f} (lower is better)")
save_result(RUN_KEY, {"model": MODEL_NAME, "use_rag": USE_RAG, "normalized_edit_distance": score,
                       "n_train_examples": 200, "examples": examples_out[:5]})

free_gpu_memory(model, trainer)


### 5.4 Qwen3.5-2B + RAG (thinking mode OFF, then ON as a second sub-test)

Note: `Qwen/Qwen3.5-2B` is the correct Hugging Face repo ID for the released (post-trained) checkpoint,
verified directly against the model card as of this notebook\'s writing -- Qwen3.5 released after my
training cutoff, so this was checked live rather than recalled. If this repo ID has since changed,
check https://huggingface.co/Qwen for the current name before running.


In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-2B"
RUN_KEY_BASE = "qwen3.5-2b-rag"
USE_RAG = True

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto",
                                              trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)

train_dataset = make_training_dataset(tokenizer, n_examples=200, use_rag=USE_RAG)

# output_dir was hardcoded to /content/checkpoints/... (Colab-only) -- now LOCAL_WORK_DIR
sft_config = SFTConfig(output_dir=str(LOCAL_WORK_DIR / "checkpoints" / RUN_KEY_BASE), num_train_epochs=3, per_device_train_batch_size=2,
                        gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=10, save_strategy="no",
                        report_to="none", max_seq_length=1024, dataset_text_field="text")
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, tokenizer=tokenizer)
trainer.train()

# Test BOTH thinking-mode settings against the same fine-tuned weights -- this is the open question
# from the handoff to AI/CV Engineer: does visible reasoning help retrieval-conditioned restoration or not.
score_no_think, examples_no_think = score_model(model, tokenizer, use_rag=USE_RAG, enable_thinking=False)
print(f"{RUN_KEY_BASE} (thinking OFF) normalized edit distance: {score_no_think:.4f}")
save_result(f"{RUN_KEY_BASE}-thinkOFF", {"model": MODEL_NAME, "use_rag": USE_RAG, "enable_thinking": False,
                                          "normalized_edit_distance": score_no_think, "examples": examples_no_think[:5]})

score_think, examples_think = score_model(model, tokenizer, use_rag=USE_RAG, enable_thinking=True, max_new_tokens=384)
print(f"{RUN_KEY_BASE} (thinking ON) normalized edit distance: {score_think:.4f}")
save_result(f"{RUN_KEY_BASE}-thinkON", {"model": MODEL_NAME, "use_rag": USE_RAG, "enable_thinking": True,
                                         "normalized_edit_distance": score_think, "examples": examples_think[:5]})

free_gpu_memory(model, trainer)


### 5.5 Qwen3.5-4B + RAG

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
RUN_KEY = "qwen3.5-4b-rag"
USE_RAG = True

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto",
                                              trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)

train_dataset = make_training_dataset(tokenizer, n_examples=200, use_rag=USE_RAG)

# output_dir was hardcoded to /content/checkpoints/... (Colab-only) -- now LOCAL_WORK_DIR
sft_config = SFTConfig(output_dir=str(LOCAL_WORK_DIR / "checkpoints" / RUN_KEY), num_train_epochs=3, per_device_train_batch_size=2,
                        gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=10, save_strategy="no",
                        report_to="none", max_seq_length=1024, dataset_text_field="text")
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset, tokenizer=tokenizer)
trainer.train()

score, examples_out = score_model(model, tokenizer, use_rag=USE_RAG, enable_thinking=False)
print(f"{RUN_KEY} normalized edit distance: {score:.4f} (lower is better)")
save_result(RUN_KEY, {"model": MODEL_NAME, "use_rag": USE_RAG, "normalized_edit_distance": score,
                       "n_train_examples": 200, "examples": examples_out[:5]})

free_gpu_memory(model, trainer)


## 8. Results summary -- run after all five (six, counting thinking on/off) sections above have completed

In [ ]:
results = load_results()

print(f"{'Run':<28}{'Normalized Edit Distance':<28}{'RAG':<8}")
print("-" * 64)
for key, r in sorted(results.items(), key=lambda kv: kv[1]["normalized_edit_distance"]):
    print(f"{key:<28}{r['normalized_edit_distance']:<28.4f}{str(r['use_rag']):<8}")

print("\nLower normalized edit distance = better restoration accuracy.")
print("Compare RAG vs no-RAG, and check whether smaller models + RAG beat the 7B fine-tune-only baseline.")
print("\nFull results with example predictions saved at:", RESULTS_PATH)
# Was a Colab-only files.download() instruction pointing at a hardcoded
# /content/ path -- results already sit at RESULTS_PATH on local disk,
# no download step needed either way.
print(f"Open it directly at: {RESULTS_PATH}")
